In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# Tarea 1 — Recolección e inventario de activos (Individual/Diagnóstico)
## 1. Cargar datasets

In [ ]:
dataset_limpio = pd.read_csv('../data/raw/movilidad_sensores_LIMPIO.csv')
dataset_contaminado = pd.read_csv('../data/raw/movilidad_sensores_CONTAMINADO.csv')
dataset_json = pd.read_json('../data/raw/clima_api_log.json')

In [ ]:
# Descripcion del dataset contaminado
dataset_contaminado.describe(include="all")

In [ ]:
# Información de las columnas
dataset_contaminado.info()

In [ ]:
# Descripcion del dataset en JSON
dataset_json.describe(include="all")

In [ ]:
# Información de los datos semiestructurados
dataset_json.info()

# 1.2 - Inventario por variable CSV
* **Dataset:** movilidad_sensores

* **Formato de origen:** CSV

* **Tipo de fuente:** Estructurada

| Variable        | Tipo de dato |
|-----------------|--------------|
|sensor_id        | nominal      |
|ubicacion        | nominal      |
|tipo_via         | nominal      |
|timestamp        | fecha        |
|conteo_vehiculos | discreto     |
|temperatura_c    | continuo     |
|condicion_clima  | nominal      |
|lat              | geoespacial (continua) |
|lon              | geoespacial (continua) |

# 1.3 - Inventario por variable JSON
* **Dataset:** clima_api_log

* **Formato de origen:** JSON

* **Tipo de fuente:** Semi estructurada

| Variable        | Tipo de dato |
|-----------------|--------------|
|request_id       | nominal      |
|timestamp        | fecha        |
|location         | Diccionario anidado de ubicación|
|weather          | Diccionario anidado de clima|


In [ ]:
# Aplanar campos anidados dentro del JSON


# location y weather son los campos que poseen diccionarios anidados.
# Se utiliza json_normalize para aplanar cada campo de forma individual

location_data = pd.json_normalize(dataset_json['location'], sep='_')
weather_data = pd.json_normalize(dataset_json['weather'], sep='_')

# Combinando los datos a través de la concatenación de los dos dataframes para normalizar la información
dataset_json_normalizado = pd.concat([
    dataset_json[['request_id', 'timestamp']],
    location_data,
    weather_data
], axis=1)

dataset_json_normalizado

## Pregunta de reflexión

**¿Qué información se pierde o se distorsiona al forzar una fuente no estructurada/semi-estructurada dentro de una tabla rectangular?**

**R:** Al aplanar la información, se está perdiendo la identificación de los campos que existen dentro de los diccionarios anidados; elementos como "weather" y "location" describen el propósito de la información y el por qué son estructuras separadas en primer lugar. Para términos de procesamiento de datos, aplanarlos es necesario para el cruce de información, pero con la negativa de perder la descripción o identificación de los campos.

# Tarea 2 — Diagnóstico GIGO sobre el archivo contaminado

In [ ]:
# Verificación de valores nulos
df = dataset_contaminado
print("Valores nulos por columna")
print(dataset_contaminado.isnull().sum())
print(f"\nTotal de valores nulos en el dataset: {df.isnull().sum().sum()}")

In [ ]:
dataset_contaminado['condicion_clima'].unique()

In [ ]:
dataset_contaminado['timestamp'].unique()

In [ ]:
dataset_contaminado[dataset_contaminado['lat'] < 0]

In [ ]:
# Datos duplicados segun la primera repeticion de cada fila segun sensor_id y timestamp
dataset_contaminado.duplicated(subset=['sensor_id','timestamp']).sum()

In [ ]:
# Tabla de duplicados segun todas las ocurrencias segun sensor_id y timestamp
dataset_contaminado[dataset_contaminado.duplicated(subset=['sensor_id','timestamp'], keep=False)]

  |Problema detectado| Columna(s) afectada(s) |	Método de detección en pandas |	¿Por qué es un riesgo para la decisión de negocio?|
|-----------------|--------------|----------------|------------|
| Consistencia de categorias | condicion_clima | df['condicion_clima'].unique() | Segrega la información de manera inadecuada, tomando valores de la misma categoría como distintos, generando resultados estadísticos inconsistentes para cada sensor. |
| Fechas mixtas | timestamp | df['timestamp'].unique() | La presencia de variedad del formato de fechas introduce errores para el análisis del comportamiento de los sensores, vehículos y demás variables según su orden temporal|
| Valores imposibles | conteo_vehiculos | df.describe() | Valores incorrectos en la medida pueden afectar medidas de tendencia central, que permiten tomar decisiones según el comportamiento de los vehículos |
| Completitud | conteo_vehiculos, temperatura_c, condicion_clima | df.isnull().sum() | La falta de alguna de las tres variables especificadas, imposibilitan encontrar la relación del comportamiento de las variables; el conteo de vehículos y la condición del clima, por ejemplo, describen como afecta la cantidad de vehiculos segun las condiciones climaticas.  |
| Error de georeferenciacion | lat, lon  | df.describe(), df[df['lat'] < 0] | Altera el contexto y referenciación geográfica de los eventos de movilidad. |
| Unicidad | Todas | df[df.duplicated(keep=True)] | Los duplicados exactos pueden sesgar la información, afectando medidas centrales |
| Unicidad (eventos de negocio) | sensor_id, timestamp | df[df.duplicated(subset=['sensor_id', 'timestamp'], keep=False)] | Se presentan duplicados en un sensor y tiempo iguales, pero con demás variables distintas, dificulta la elección de información relevante de los datos y su certeza. |

# Tarea 3 - Transformación y limpieza con pandas

### 3.1 Consistencia de categorias

Para las inconsistencias de mayusculas y minusculas, se pasan todas hacia minúscula, para luego obtener la lista de valores únicos existentes en la columna:

soleado, sol, nubes, nublado, lluvioso, lluvia, nan

Para los valores categóricos que significan lo mismo (sol, soleado), se redujo a una sola para cada uno de estos

* **soleado - sol:** se elige soleado
* **nublado - nubes:** se elige nublado
* **lluvioso - lluvia:** se elige lluvioso

Para los valores NaN, y además de tratarse de un valor categórico, se decidió utilizar la **moda** como reemplazo para estos valores faltantes. No se decidió cortar las filas debido a que la cantidad de datos es relativamente pequeña (<1500) y alterar mas el tamaño podria afectar la calidad del estudio.

Cabe mencionar el sesgo que esto introduce a los datos, pero es un riesgo a tomar dada la cantidad limitada de información

In [ ]:
import numpy as np

# 1. Consistencia de categorias (condicion_clima)

# Se obtienen las condiciones climaticas existentes de forma única, evidenciando diferentes tipeos para el mismo tipo de categoría
# Se utiliza la funcion lower para descartar de primera mano las diferecias por mayusculas y minusculas
dataset_contaminado['condicion_clima'] = dataset_contaminado['condicion_clima'].str.lower()
# Diferencias:

# soleado - sol: se elige soleado
# nublado - nubes: se elige nublado
# lluvioso - lluvia: se elige lluvia

dataset_contaminado['condicion_clima'].replace("sol", "soleado", inplace=True)
dataset_contaminado['condicion_clima'].replace("lluvia", "lluvioso", inplace=True)
dataset_contaminado['condicion_clima'].replace("nubes", "nublado", inplace=True)

### 3.2 Fechas mixtas

Respecto a fechas mixtas, existen 5 tipos distintos de formato de fechas, analizados y obtenidos previamente con el comando de unique. Estas tienen un formato que sigue de la siguiente forma:

  1. Estandar          -> "2025-03-09 16:00:00"
  2. ISO con zona (Z)  -> "2025-03-09T06:00:00Z"
  3. Dia/Mes/Anio      -> "13/03/2025 02:00"
  4. Epoch (segundos)  -> "1740880800"
  5. Texto hibrido     -> "06 de March de 2025, 22:00"

Se decidió convertir todos los datos (los cuales no poseian nulos, sino maneras de conversión distintas para cada tipo de fecha, como Epoch) hacia fechas que sigan el estandar **ISO 8601**

In [ ]:
def normalizar_fecha(val):
    cadena = str(val).strip()

    # Caso 4: Si es un número puro, es Epoch en segundos
    if cadena.isdigit():
        return pd.to_datetime(int(cadena), unit='s').replace(tzinfo=None)

    # Caso 5: Limpiar el texto híbrido ("de" y comas)
    if ' de ' in cadena:
        cadena = cadena.replace(' de ', ' ').replace(',', '')
        return pd.to_datetime(cadena, format='%d %B %Y %H:%M', errors='coerce').replace(tzinfo=None)

    # Casos 1, 2 y 3: Formatos manejados automáticamente por Pandas
    return pd.to_datetime(cadena, format='mixed', errors='coerce').replace(tzinfo=None)

# 1. Aplicar la función de conversión fila por fila
dataset_contaminado['timestamp'] = dataset_contaminado['timestamp'].apply(normalizar_fecha)

dataset_contaminado['timestamp'].unique()

### 3.3 Tratamiento de Valores Imposibles (Outliers)
**Técnica utilizada:** Identificación mediante Rango Intercuartílico (IQR) y conversión a `NaN`.

*   **¿Por qué esta técnica y no la eliminación?**: La eliminación de filas completas conlleva una pérdida de información valiosa en otras variables (como clima o ubicación, etc). Al convertir los valores atípicos (negativos o extremos como el `99999`) en `NaN`, aislamos el error sin sacrificar el resto del registro.
*   **¿Por qué IQR y no un umbral fijo?**: El método IQR es estadísticamente objetivo y se adapta a la distribución real de los datos. Esto evita sesgos humanos al definir qué es "demasiado alto".
* **¿Por qué no imputamos directamente con el promedio?**: Este enfoque de limpiar antes de imputar garantiza que los valores nulos generados por errores de sistema sean reemplazados por valores coherentes con la realidad estadística del entorno, también evitamos que al momento de hallar la media para imputar en la completitud no esté sesgada. Igualmente, estos NaN se imputan en la completitud.

In [ ]:
# Calculamos los cuartiles y el IQR
Q1 = dataset_contaminado['conteo_vehiculos'].quantile(0.25)
Q3 = dataset_contaminado['conteo_vehiculos'].quantile(0.75)
IQR = Q3 - Q1

# Definimos el límite superior para outliers
limite_superior = Q3 + 1.5 * IQR

# Aplicamos la limpieza:
# Los que superen el límite superior o sean negativos los pasamos a NaN
mask_outliers = (dataset_contaminado['conteo_vehiculos'] < 0) | (dataset_contaminado['conteo_vehiculos'] > limite_superior)
dataset_contaminado.loc[mask_outliers, 'conteo_vehiculos'] = np.nan

### 3.4 Completitud e Imputación de Datos
**Técnica utilizada:** Imputación por mediana (Numéricos) y moda (Categóricos).

#### Variables Numéricas (`conteo_vehiculos`, `temperatura_c`)
*   **¿Por qué Mediana en lugar de Media?**: La media es altamente sensible a valores extremos. En datos de sensores donde pueden existir fallos, la **mediana** ofrece una representación mucho más fiel del valor central, garantizando que los datos imputados no distorsionen la tendencia general.

#### Variable Categórica (`condicion_clima`)
*   **¿Por qué Moda?**: Para datos no numéricos, no es posible calcular promedios. La **moda** es el estadístico de tendencia central más adecuado, ya que asume el escenario más probable basado en la frecuencia histórica del dataset.

In [ ]:
# Rellenar NaNs en 'conteo_vehiculos' y 'temperatura_c' con la mediana.
dataset_contaminado['conteo_vehiculos'].fillna(dataset_contaminado['conteo_vehiculos'].median(), inplace=True)
dataset_contaminado['temperatura_c'].fillna(dataset_contaminado['temperatura_c'].median(), inplace=True)

# Rellenar NaNs en 'condicion_clima' con la moda.
mode_condicion_clima = dataset_contaminado['condicion_clima'].mode()[0]
dataset_contaminado['condicion_clima'].fillna(mode_condicion_clima, inplace=True)

### 3.5 Error Georeferenciación
Se tomaron las Coordenadas    6°15′0.72″ N, 75°34′3.31″ W    En decimal    6.2502°, -75.567585° como el espacio georeferencial valido para la ciudad, y en la fase de analisis de los datos se notó un comportamiento en las varables "lat" y "lot".

Se detectaron errores de digitación, ya que varios de los datos de georeferenciación en estas columnas estaban trocados, por lo cual se hizo una validacion simple en estas columnas basado en los rangos validos y se hizo un swap en las que era necesario.

La indicación de valores "válidos" para Medellín se obtuvo a partir de la siguiente web de datos georeferenciales:

https://geohack.toolforge.org/geohack.php?language=es&pagename=Medell%C3%ADn&params=6.250200154879_N_-75.567584500697_E_type:city

In [ ]:
# Define a reasonable bounding box for Medellín based on the provided coordinates (6.2502° N, 75.567585° W)
MEDELLIN_LAT_MIN, MEDELLIN_LAT_MAX = 6.0, 6.5 # Approximately around 6.25
MEDELLIN_LON_MIN, MEDELLIN_LON_MAX = -76.0, -75.0 # Approximately around -75.56

def correct_geolocation(row):
    lat = row['lat']
    lon = row['lon']

    # Heuristic to detect and correct swapped lat/lon values
    # This assumes that if lat is in the longitude range and lon is in the latitude range, they are swapped.
    is_swapped = (MEDELLIN_LON_MIN <= lat <= MEDELLIN_LON_MAX) and \
                 (MEDELLIN_LAT_MIN <= lon <= MEDELLIN_LAT_MAX)

    if is_swapped:
        # Perform the swap
        corrected_lat = lon
        corrected_lon = lat
    else:
        # No swap, use original values
        corrected_lat = lat
        corrected_lon = lon

    # Now, check if the (potentially corrected) coordinates are within Medellín's bounds
    if not (MEDELLIN_LAT_MIN <= corrected_lat <= MEDELLIN_LAT_MAX and \
            MEDELLIN_LON_MIN <= corrected_lon <= MEDELLIN_LON_MAX):
        return pd.Series({'lat': np.nan, 'lon': np.nan})
    else:
        return pd.Series({'lat': corrected_lat, 'lon': corrected_lon})

# Apply the correction to 'lat' and 'lon' columns
dataset_contaminado[['lat', 'lon']] = dataset_contaminado.apply(correct_geolocation, axis=1)

### 3.6 Unicidad (todas las columnas)
Para esto, se eliminaron por completo las copias de los registros con duplicados exactos, evitando ruido y variaciones en el procesamiento y analisis. Son datos totalmente repetidos, por lo que no poseen algun valor y se le ha realizado drop

In [ ]:
dataset_contaminado.drop_duplicates(inplace=True)

### 3.7 Unicidad (eventos de negocio: sensor_id, timestamp)
Analizando los datos, se reporta que la duplicidad de eventos de negocio podria presentarse con las variables 'sensor_id','timestamp', ya que si en varios registros estas variables tenian valores iguales, las demas columnas no podrian variar, ya que seria imposible que el mismo sensor en la misma fecha y hora detectara algo diferente.

**Con esta información presente:**

Función utilizada `dataset_contaminado[dataset_contaminado.duplicated(subset=['sensor_id','timestamp'], keep=False)]`

Para confirmar la precencia de este tipo de duplicidad.

En la imputacion de este error usamos la funcion: `dataset_contaminado = dataset_contaminado.drop_duplicates(subset=['sensor_id', 'timestamp'], keep='first')`

Con el subconjunto de las dos variables identificadas antes como argumento, y el argumento **keep = first** para que tome la primera coincidencia y elimine los registros que generan duplicidad.

In [ ]:
dataset_procesado = dataset_contaminado.drop_duplicates(subset=['sensor_id', 'timestamp'], keep='first')

dataset_procesado.to_csv("../data/processed/dataset_procesado.csv", index=False)

# 4 — Analítica descriptiva cuantitativa, cualitativa y gráfica

#### 4.1 **Cuantitativa:** Medidas de tendencia central y dispersión para al menos 2 variables continuas y 1 discreta, con la medida correcta según su distribución


##### Medidas segun la distribución de las variables


*   **conteo_vehiculos:** Mediana. Se utiliza la mediana debido a que es una medida discreta, con conteos de numeros naturales. Aunque la media para esta variable es *matemáticamente* correcta, su naturaleza es discreta.
*   **temperatura_c:** Media. Al ser variable continua, se puede promediar la temperatura capturada a través del tiempo
*   **lat, lon:** Media. Estas variables también son continuas y, por ende, pueden ser promediadas. No obstante, el promedio de estas variables no deben promediarse utilizando la forma ingenua. [Se usan métodos especiales como se muestra en este libro](http://palaeo.spb.ru/pmlibrary/pmbooks/mardia&jupp_2000.pdf). Para el ámbito de este trabajo, se realiza la media ingenua.


In [ ]:
variables_df = ["conteo_vehiculos", "temperatura_c", "lat", "lon"]

comparativa = pd.DataFrame({
    "Variable": variables_df,
    "Media": [dataset_procesado[var].mean() for var in variables_df],
    "Mediana": [dataset_procesado[var].median() for var in variables_df],
    "Moda": [dataset_procesado[var].mode().values[0] if dataset_procesado[var].mode().size > 0 else None for var in variables_df],
    "Desviación Estándar": [dataset_procesado[var].std() for var in variables_df],
    "Rango": [dataset_procesado[var].max() - dataset_procesado[var].min() for var in variables_df],
    "Asimetría": [dataset_procesado[var].skew() for var in variables_df],
    "Tipo de medida": ['Mediana', 'Media', 'Media', 'Media']
})


comparativa

#### 4.2 **Cualitativa**
 * **Variables categóricas**: condicion_clima, ubicacion


##### Tabla de frecuencia - Ubicación
 ![Ubicacion](https://i.imgur.com/CFrDFRo.png)
 Se puede observar la distribución de los valores de la variable "ubicacion", podemos identificar que los datos fueron captados de manera prácticamente equitativa en cada ubicación, teniendo así la misma frecuencia y porcentaje (proporción).

 ##### Tabla de frecuencia - Condición climatica
 ![Condicion_clima](https://i.imgur.com/nJlqRAZ.png)

 Se puede observar la distribución de los valores de la variable condicion_clima, identificando que la mayoría de los datos fueron tomados en días soleados, seguido de nublados y por último lluviosos.

In [ ]:
# 1. Calcular frecuencias, proporciones y acumulados
clima_counts = dataset_procesado['condicion_clima'].value_counts().reset_index()
clima_counts.columns = ['Categoría', 'Frecuencia']
total = clima_counts['Frecuencia'].sum()
clima_counts['Porcentaje'] = (clima_counts['Frecuencia'] / total * 100).round(1)
clima_counts['Porcentaje Acumulado'] = clima_counts['Porcentaje'].cumsum().round(1)

# 2. Fila de Total
fila_total = pd.DataFrame({
    'Categoría': ['Total'],
    'Frecuencia': [total],
    'Porcentaje': [100.0],
    'Porcentaje Acumulado': [100.0]
})
tabla_final = pd.concat([clima_counts, fila_total], ignore_index=True)
n_filas = len(tabla_final)

# 3. Generar colores en función del valor (heatmap), usando el Porcentaje
#    Mientras más alto el porcentaje, más intenso el color azul
colorscale = px.colors.sequential.Blues
valores_norm = (clima_counts['Porcentaje'] / clima_counts['Porcentaje'].max()).tolist()
colores_valor = px.colors.sample_colorscale(colorscale, valores_norm)
colores_valor.append('rgb(142,169,219)')  # color fijo, más oscuro, para resaltar el Total

# 4. Texto blanco cuando el color de fondo es oscuro (mejor contraste)
def color_texto(color_rgb):
    # sample_colorscale devuelve 'rgb(r,g,b)'
    rgb = color_rgb.replace('rgb(', '').replace(')', '').split(',')
    r, g, b = [int(float(x)) for x in rgb]
    brillo = (r*299 + g*587 + b*114) / 1000
    return 'white' if brillo < 140 else 'black'

colores_fuente = [color_texto(c) for c in colores_valor]

# 5. Definir la tabla en Plotly
fig = go.Figure(data=[go.Table(
    columnwidth=[100, 100, 90, 130],
    header=dict(
        values=['<b>Categoría</b>', '<b>Frecuencia</b>', '<b>Porcentaje</b>', '<b>% Acumulado</b>'],
        fill_color='navy',
        align='center',
        font=dict(color='white', size=14),
        height=35
    ),
    cells=dict(
        values=[
            tabla_final['Categoría'],
            tabla_final['Frecuencia'].map('{:,.0f}'.format),
            tabla_final['Porcentaje'].map('{:.1f}%'.format),
            tabla_final['Porcentaje Acumulado'].map('{:.1f}%'.format)
        ],
        fill_color=[colores_valor] * 4,        # mismo color para toda la fila
        font=dict(color=[colores_fuente] * 4, size=12),
        align='center',
        height=30
    )
)])

# 6. Ajustar altura dinámicamente para que no sobre espacio en blanco
altura_total = 35 + (n_filas * 30) + 60  # header + filas + título/márgenes
fig.update_layout(
    title_text='Tabla de Frecuencias: Condición Climática',
    title_x=0.5,
    height=altura_total,
    margin=dict(l=10, r=10, t=50, b=10)
)
fig.show()

In [ ]:
# 1. Calcular frecuencias, proporciones y acumulados
ubicacion_counts = dataset_procesado['ubicacion'].value_counts().reset_index()
ubicacion_counts.columns = ['Categoría', 'Frecuencia']
total = ubicacion_counts['Frecuencia'].sum()
ubicacion_counts['Porcentaje'] = (ubicacion_counts['Frecuencia'] / total * 100).round(1)
ubicacion_counts['Porcentaje Acumulado'] = ubicacion_counts['Porcentaje'].cumsum().round(1)

# 2. Fila de Total
fila_total = pd.DataFrame({
    'Categoría': ['Total'],
    'Frecuencia': [total],
    'Porcentaje': [100.0],
    'Porcentaje Acumulado': [100.0]
})
tabla_final = pd.concat([ubicacion_counts, fila_total], ignore_index=True)
n_filas = len(tabla_final)

# 3. Generar colores en función del valor (heatmap), usando el Porcentaje
colorscale = px.colors.sequential.Greens
valores_norm = (ubicacion_counts['Porcentaje'] / ubicacion_counts['Porcentaje'].max()).tolist()
colores_valor = px.colors.sample_colorscale(colorscale, valores_norm)
colores_valor.append('rgb(56,142,60)')
# 4. Texto blanco cuando el color de fondo es oscuro (mejor contraste)
def color_texto(color_rgb):
    rgb = color_rgb.replace('rgb(', '').replace(')', '').split(',')
    r, g, b = [int(float(x)) for x in rgb]
    brillo = (r*299 + g*587 + b*114) / 1000
    return 'white' if brillo < 140 else 'black'

colores_fuente = [color_texto(c) for c in colores_valor]

# 5. Definir la tabla en Plotly
fig = go.Figure(data=[go.Table(
    columnwidth=[130, 100, 90, 130],
    header=dict(
        values=['<b>Categoría</b>', '<b>Frecuencia</b>', '<b>Porcentaje</b>', '<b>% Acumulado</b>'],
        fill_color='darkgreen',
        align='center',
        font=dict(color='white', size=14),
        height=35
    ),
    cells=dict(
        values=[
            tabla_final['Categoría'],
            tabla_final['Frecuencia'].map('{:,.0f}'.format),
            tabla_final['Porcentaje'].map('{:.1f}%'.format),
            tabla_final['Porcentaje Acumulado'].map('{:.1f}%'.format)
        ],
        fill_color=[colores_valor] * 4,
        font=dict(color=[colores_fuente] * 4, size=12),
        align='center',
        height=30
    )
)])

# 6. Ajustar altura dinámicamente
altura_total = 35 + (n_filas * 30) + 60
fig.update_layout(
    title_text='Tabla de Frecuencias: Ubicación',
    title_x=0.5,
    height=altura_total,
    margin=dict(l=10, r=10, t=50, b=10)
)
fig.show()

### Tabla de contingencia
* **Variable categórica**: condicion_clima
* **Variable ordinal**: conteo_vehiculos

En el dataset no hay como tal una "variable ordinal", para crear la tabla de contingencia convertimos la variable `conteo_vehiculos` en una variable ordinal.

El método que usamos para separar la variable en rangos y así tener su ordinalidad fue la siguiente:

```
categorias, bins = pd.qcut(
    dataset_procesado['conteo_vehiculos'],
    q=3,
    labels=['bajo', 'medio', 'alto'],
    retbins=True
)
```

De esta manera, cortamos los puntos `[0, 16, 24, 54]` quedando así:
* **Tráfico Bajo**: [0, 16)
* **Tráfico Medio**: (16, 24]
* **Tráfico Alto**: (24, 54]

![tabla_contingencia](https://i.imgur.com/0rGIdYS.png)

In [ ]:
# 1. Crear la variable ordinal aparte, sin agregarla al dataset
categorias, bins = pd.qcut(
    dataset_procesado['conteo_vehiculos'],
    q=3,
    labels=['bajo', 'medio', 'alto'],
    retbins=True
)

# 2. Calcular la tabla de contingencia con márgenes de Total incluidos
contingency_tab = pd.crosstab(
    dataset_procesado['condicion_clima'],
    categorias,
    margins=True,
    margins_name='Total'
).reset_index().rename(columns={'condicion_clima': 'Condición Clima'})

n_filas = len(contingency_tab)
columnas_datos = ['bajo', 'medio', 'alto', 'Total']

# 3. Generar colores tipo heatmap por celda (más oscuro = más observaciones)
colorscale = px.colors.sequential.Purples
max_valor = contingency_tab.loc[contingency_tab['Condición Clima'] != 'Total', columnas_datos[:-1]].values.max()

def color_celda(valor, es_total_fila_col=False):
    if es_total_fila_col:
        return 'rgb(120,100,160)'  # color fijo para totales
    norm = valor / max_valor
    return px.colors.sample_colorscale(colorscale, norm)[0]

# 4. Texto blanco o negro según qué tan oscuro sea el fondo (contraste automático)
def color_texto(color_rgb):
    rgb = color_rgb.replace('rgb(', '').replace(')', '').split(',')
    r, g, b = [int(float(x)) for x in rgb]
    brillo = (r*299 + g*587 + b*114) / 1000
    return 'white' if brillo < 140 else 'black'

colores_por_columna = []
fuentes_por_columna = []
for col in columnas_datos:
    colores_col = []
    fuentes_col = []
    for i, row in contingency_tab.iterrows():
        es_total = (row['Condición Clima'] == 'Total') or (col == 'Total')
        color = color_celda(row[col], es_total)
        colores_col.append(color)
        fuentes_col.append(color_texto(color))
    colores_por_columna.append(colores_col)
    fuentes_por_columna.append(fuentes_col)

    # 5. Color y fuente de la primera columna, ahora acorde a la paleta morada
colores_primera_col = [
    'rgb(120,100,160)' if v == 'Total' else 'rgb(237,231,246)'  # morado clarito, misma familia de color
    for v in contingency_tab['Condición Clima']
]
fuentes_primera_col = ['white' if v == 'Total' else 'black' for v in contingency_tab['Condición Clima']]

# 6. Definir la tabla en Plotly
fig = go.Figure(data=[go.Table(
    columnwidth=[130, 90, 90, 90, 90],
    header=dict(
        values=['<b>Condición Clima</b>', '<b>Tráfico Bajo</b>', '<b>Tráfico Medio</b>', '<b>Tráfico Alto</b>', '<b>Total</b>'],
        fill_color='indigo',
        align='center',
        font=dict(color='white', size=14),
        height=35
    ),
    cells=dict(
        values=[
            contingency_tab['Condición Clima'],
            contingency_tab['bajo'],
            contingency_tab['medio'],
            contingency_tab['alto'],
            contingency_tab['Total']
        ],
        fill_color=[colores_primera_col] + colores_por_columna,
        font=dict(color=[fuentes_primera_col] + fuentes_por_columna, size=12),
        align='center',
        height=32
    )
)])

# 7. Ajustar altura con más holgura para que no salga scroll
altura_total = 35 + (n_filas * 32) + 100
fig.update_layout(
    title_text='Tabla de Contingencia: Clima vs Nivel de Tráfico',
    title_x=0.5,
    height=altura_total,
    margin=dict(l=10, r=10, t=60, b=30)
)

fig.show()

### 4.3 **Gráfica**

#### Cantidad de registros por clima


![registros por clima](https://i.imgur.com/7OSlal5.png)

Se evidencia que el mayor estado climático entre la totalidad de los registros es soleado

![vehiculos_hora_ubic](https://i.imgur.com/cnCsXpj.png)

Este gráfico describe el comportamiento de los vehículos en cada ubicación según el paso de las horas. Se observan picos de cantidad de vehículos entre las 8-9 y 17-19, haciendo resaltar la comúnmente llamada "hora pico"

![distribucion_geo](https://i.imgur.com/Ifyo8Xf.png)

La distribución de la información georeferencial muestra que los sensores se encuentran dentro del rango aceptable que describe la posición de la ciudad de Medellín. Además, se evidencia la cantidad equitativa de sensores entre las ubicaciones, información previamente dada en las tablas de frecuencia (240 por ubicación)

In [ ]:
# Extraer la hora del timestamp
dataset_procesado['hora'] = dataset_procesado['timestamp'].dt.hour

# Crear una figura con 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# 1. Gráfico de barras horizontales: Cantidad de registros por clima
sns.countplot(y='condicion_clima', data=dataset_procesado, ax=axes[0], palette='viridis', hue='condicion_clima', legend=False)
axes[0].set_title('Cantidad de Registros por Clima')
axes[0].set_xlabel('Frecuencia de Registros')
axes[0].set_ylabel('Condición Climática')

# 2. Gráfico de líneas: Suma de vehículos según la hora desglosado por ubicación
sns.lineplot(x='hora', y='conteo_vehiculos', data=dataset_procesado, hue='ubicacion', ax=axes[1], marker='o', errorbar=None)
axes[1].set_title('Tendencia de Vehículos por Hora y Ubicación')
axes[1].set_xlabel('Hora (0-23)')
axes[1].set_ylabel('Frecuencia de Vehículos')
axes[1].set_xticks(range(0, 24))
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(title='Ubicación', bbox_to_anchor=(1.05, 1), loc='upper left')

# 3. Diagrama de dispersión: Latitud vs Longitud
sns.scatterplot(x='lon', y='lat', data=dataset_procesado, hue='ubicacion', ax=axes[2], palette='Set1', s=100, alpha=0.7)
axes[2].set_title('Distribución Geográfica de Sensores (Lat vs Lon)')
axes[2].set_xlabel('Longitud')
axes[2].set_ylabel('Latitud')
axes[2].grid(True, linestyle=':', alpha=0.8)
axes[2].legend(title='Ubicación', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

# 5. De la estadística a la decisión

#### **5.1** ¿Cuál es la pregunta de negocio que este análisis realmente responde?

Este análisis responde a la pregunta de **¿Cuál es la probabilidad de que un sensor detecte alta frecuencia de autos dada una combinacion de clima, horas pico y no pico, y vía?**


#### **5.2** ¿Cuál es la recomendación concreta y accionable?

Tomar los datos de tiempo, clima y ubicación que los sensores extraen, los cuales muestran en mayor medida el comportamiento de los vehículos en distintas avenidas, la frecuencia de estos según el tipo de clima (tener en cuenta vehículos como motocicletas, por ejemplo), y las congestiones que se dan en horas de la mañana y tarde. A partir de esta información, por ejemplo, organizaciones públicas responsables del tráfico en Medellín podrían tomar decisiones que mitiguen las congestiones (Reorganización y planeación en vías, categorización de vehículos permitidos según el día, etc.)

La calidad de la decisión está basada mayoritariamente por la cantidad de datos obtenidos y su alcance, límites los cuales serán explicados posteriormente.

#### **5.3** ¿Qué le costaría a la organización un Falso Positivo y un Falso Negativo en esta decisión?

**Costo de un Falso Positivo (FP):** Se predice alta congestión (se activa una intervención), pero no ocurre.

la Organizacion encargada podria tomar decisiones que no atacarian el problema, como una movilización innecesaria de policias de tránsito, ajustar semáforos que no eran críticos, enviar alertas falsas, que generan desconfianza en los usuarios.
Esas acciones generarian ineficiencia operativa y economica porque habria que destinar recursos y personal.

**Costo de un Falso Negativo (FN):** No se predice alta congestión (no se activa una intervención), pero sí ocurre.

la Organizacion encargada tomaria decisiones que no son acordes al estado real de la movilidad y congestion en la cliudad, lo cual generaria un Aumento de los tiempos de viaje, frustración ciudadana, contaminación ambiental por vehículos detenidos, retrasos en servicios esenciales (emergencias, transporte público).



#### **5.4** Limitación de los datos


Inicialmente, existe incertidumbre estadística que ninguna limpieza de datos puede eliminar. Por lo tanto, los resultados deben interpretarse como probabilidades y no como certezas absolutas al momento de tomar decisiones

**Tamaño de la muestra**

El dataset cuenta con un tamaño de muestra reducido (1,440 registros). Esto implica que las relaciones observadas entre clima, cantidad de vehículos, tipo de vía y tiempo, son estimaciones basadas en una muestra limitada, no verdades absolutas.

**Concentración temporal**

Aunque el dataset abarca fechas de marzo a diciembre de 2025, el 99% de los registros pertenece a marzo. Esto significa que el análisis refleja principalmente las condiciones de un solo mes, y las conclusiones sobre la relación entre clima y tráfico no pueden generalizarse a otras épocas del año (por ejemplo, temporada de lluvias, vacaciones, fin de año)

**Categorías de tráfico definidas estadísticamente**

Los niveles 'bajo', 'medio' y 'alto' de tráfico se definieron con base en terciles de la propia muestra (no según un umbral técnico de capacidad vial real). Esto significa que 'tráfico alto' en este análisis es relativo a los datos observados, no necesariamente representa congestión real según estándares de tránsito

**Simplificación de la realidad**

Los datos capturan solo algunas variables (clima, ubicación, cantidad de vehículos, tiempo), pero el comportamiento real del tráfico depende de muchos otros factores no controlados ni registrados (eventos, obras, accidentes, festivos, decisiones humanas). El modelo no puede reflejar toda esa complejidad, así que las conclusiones deben verse como una aproximación, no como la explicación completa del fenómeno de movilidad
